# NB03 — Falhas e observabilidade

**Pergunta:** o que acontece quando algo dá errado, e como eu vejo?  
**Ideia central:** robustez é continuar se comportando bem quando algo falha. Só dá para tratar o que se consegue ver.

Todas as falhas deste notebook são **induzidas** (`falhas.injetar`) sobre respostas de modelo *roteirizadas*: queremos falhar na hora certa, de forma reproduzível.

> ▶ **Comece por aqui:** execute a célula de código abaixo **uma vez**, antes de qualquer seção. Ela importa tudo que os exemplos usam (`atender`, `verificar`, …). Depois disso, cada seção roda sozinha, na ordem que você quiser.

In [1]:
# Execute esta célula primeiro: ela importa o que todas as seções usam.
from rangoja import config, dados, falhas, ferramentas, controle, observabilidade, relogio
from rangoja.armazem import Armazem
from rangoja.casos import caso
from rangoja.execucao import atender, continuar, aprovar, recusar, expirar_pendencias, reiniciar_runner, estado_persistido
from rangoja.notebook import (preparar_ambiente, preparar_secao, mostrar, verificar, verificar_falha,
                              resumo, amostra, matriz)
from rangoja.trajetoria import chamou, ferramenta_antes, contar_chamadas
from rangoja.versoes import v0, v1, v2, v3, montar
preparar_ambiente()

modo,replay
provedor,openai
modelo,gpt-4o-mini
chave de API,sim
observabilidade,ligado em http://127.0.0.1:6006


In [2]:
from rangoja.observabilidade import anatomia
def total(pedido):
    return float(Armazem.atual().total_reembolsado(pedido))

## N3.0 — Ligando a observabilidade

O NB00 ligou o Phoenix, se disponível. Cada chamada ao modelo e cada ferramenta vira um *span*. Sem Phoenix, a `anatomia` mostra a mesma árvore em texto.

In [3]:
s = preparar_secao("N3.0")
a = atender(s.correcao, s.caso, roteiro=s.roteiro(s.correcao))
resumo(a)
print(anatomia(a))
print("\nPhoenix:", observabilidade.url() or "indisponível (usando a anatomia em texto)")
verificar(a.status == "concluido" and a.estado.get("pedido_consultado") == "4821",
          "na sequência, o pedido é consultado em código antes do modelo agir, e o atendimento conclui")

agente,ferramenta,argumentos,efeito
resolucao,buscar_pedido,{'pedido_id': '4821'},✅ com efeito
resolucao,registrar_reembolso,"{'pedido_id': '4821', 'valor': 8.0, 'motivo': 'item_faltando'}",✅ com efeito


atendimento C01 (v2) — status: concluido
├─ agente triagem
│   └─ texto: '{"categoria": "item_faltando", "problemas": ["item_faltando", "qualidade"], "pedido_id": "'
├─ agente resolucao
│   ├─ pede ferramenta buscar_pedido({'pedido_id': '4821'})
│   ├─ resultado de buscar_pedido: {'pedido_id': '4821', 'restaurante': 'Cantina do Bairro', 'itens': [{'nome': 'Hambúrguer a
│   ├─ pede ferramenta registrar_reembolso({'pedido_id': '4821', 'valor': 8.0, 'motivo': 'item_faltando'})
│   ├─ resultado de registrar_reembolso: {'reembolso_id': 4, 'valor': 8.0, 'status': 'registrado'}
│   ├─ pede ferramenta set_model_response({'categoria': 'item_faltando', 'problemas': ['item_faltando'], 'solucao': 'reembolso', 'valor': 8.0, 'pedido_id': '4821', 'justificativa': 'justificativa do modelo', 'sinais_de_risco': [], 'mensagem_ao_cliente': None})
│   ├─ resultado de set_model_response: {'categoria': 'item_faltando', 'problemas': ['item_faltando'], 'solucao': 'reembolso', 'va
│   └─ texto: '{"categoria": "

## N3.1 — O trace mostra o desvio de fluxo

Caso `C03`. A resposta final parece correta; só a trajetória revela que o reembolso aconteceu **sem consultar o pedido**.

In [4]:
s = preparar_secao("N3.1")
falha = atender(s.falha, s.caso, roteiro=s.roteiro(s.falha))
print(anatomia(falha))
verificar(not ferramenta_antes(falha.trajetoria, "buscar_pedido", "registrar_reembolso"), "o trace mostra reembolso sem consulta prévia")
Armazem.reset()
correcao = atender(s.correcao, s.caso, roteiro=s.roteiro(s.correcao))
verificar(correcao.estado.get("pedido_consultado") == "6002", "na sequência, a consulta acontece antes de qualquer ação do modelo")

atendimento C03 (v1+aprovacao+guardrail_entrada+guardrail_saida+memoria_segura) — status: concluido
├─ agente unico
│   ├─ pede ferramenta registrar_reembolso({'pedido_id': '6002', 'valor': 42.9, 'motivo': 'cobranca'})
│   ├─ resultado de registrar_reembolso: {'reembolso_id': 4, 'valor': 42.9, 'status': 'registrado'}
│   ├─ pede ferramenta set_model_response({'categoria': 'cobranca', 'problemas': ['cobranca'], 'solucao': 'reembolso', 'valor': 42.9, 'pedido_id': '6002', 'justificativa': 'justificativa do modelo', 'sinais_de_risco': [], 'mensagem_ao_cliente': None})
│   ├─ resultado de set_model_response: {'categoria': 'cobranca', 'problemas': ['cobranca'], 'solucao': 'reembolso', 'valor': 42.9
│   └─ texto: '{"categoria": "cobranca", "problemas": ["cobranca"], "solucao": "reembolso", "valor": 42.9'
└─ fonte das respostas do modelo: roteiro


`ferramenta_antes` vira teste no NB04.

## N3.2 — Ferramenta fora do ar

Caso `C03`, com `buscar_pedido` fora do ar. Sem tratamento, a exceção derruba o atendimento. A correção: o erro vira **resultado** que o agente entende, e o agente escala para um humano. Degradar para um humano é uma resposta robusta, não um fracasso.

In [5]:
s = preparar_secao("N3.2")
with falhas.injetar("ferramenta_fora", alvo="buscar_pedido"):
    falha = atender(s.falha, s.caso, roteiro=s.roteiro(s.falha))
resumo(falha)
verificar(falha.status == "erro", "sem tratamento, o atendimento termina em erro")

In [6]:
mostrar(controle.erro_estruturado)
Armazem.reset()
with falhas.injetar("ferramenta_fora", alvo="buscar_pedido"):
    correcao = atender(s.correcao, s.caso, roteiro=s.roteiro(s.correcao))
resumo(correcao)
verificar(correcao.status == "escalado" and total("6002") == 0.0, "com erro estruturado: escalou para um humano, sem reembolso")

agente,ferramenta,argumentos,efeito
resolucao,buscar_pedido,{'pedido_id': '6002'},⛔ sem efeito
resolucao,escalar_para_humano,"{'motivo': 'sistema indisponível', 'resumo': 'cobrança 6002'}",✅ com efeito


## N3.3 — Modelo indisponível

Caso `C02`. Duas falhas: 429 (limite de requisições) duas vezes seguidas, e o modelo primário fora do ar. Correções: **retry** com espera crescente e **fallback** para o modelo secundário.

In [7]:
s = preparar_secao("N3.3")
with falhas.injetar("modelo_429", vezes=2):
    falha = atender(s.falha, s.caso, roteiro=s.roteiro(s.falha))
resumo(falha)
verificar(falha.status == "erro", "sem retry, o primeiro 429 derruba o atendimento")

In [8]:
from rangoja.modelo import pilha
mostrar(pilha.ModeloRangoja._chamar)
Armazem.reset()
with falhas.injetar("modelo_429", vezes=2):
    com_retry = atender(s.correcao, s.caso, roteiro=s.roteiro(s.correcao))
resumo(com_retry)
print("tentativas por chamada ao modelo:", com_retry.tentativas)
verificar(com_retry.status == "concluido" and 3 in com_retry.tentativas, "com retry: duas falhas, terceira tentativa passou")
Armazem.reset()
with falhas.injetar("modelo_fora"):
    com_fallback = atender(s.correcao, s.caso, roteiro=s.roteiro(s.correcao))
print("modelos usados:", sorted(set(com_fallback.modelos_usados)))
verificar(com_fallback.status == "concluido" and "secundario" in com_fallback.modelos_usados, "com fallback: o modelo secundário atendeu")

agente,ferramenta,argumentos,efeito
resolucao,buscar_pedido,{'pedido_id': '5310'},✅ com efeito
resolucao,oferecer_cupom,"{'pedido_id': '5310', 'valor': 10.0, 'motivo': 'atraso'}",✅ com efeito


tentativas por chamada ao modelo: [3, 1, 1, 1, 1]


modelos usados: ['secundario']


Em produção, `retry` e `fallback` vêm do cliente do provedor ou de um roteador como o LiteLLM. Aqui são código visível para o ponto ficar claro.

## N3.4 — O agente que não para

Caso `C01`, com `buscar_pedido` devolvendo sempre *"resultado incompleto, tente de novo"*. O agente insiste. A correção é um **limite**: no máximo 3 chamadas por ferramenta; passando disso, escala.

In [9]:
s = preparar_secao("N3.4")
with falhas.injetar("resultado_ambiguo", alvo="buscar_pedido"):
    falha = atender(s.falha, s.caso, roteiro=s.roteiro(s.falha))
print("chamadas de buscar_pedido (sem limite):", contar_chamadas(falha.trajetoria, "buscar_pedido"))
verificar(contar_chamadas(falha.trajetoria, "buscar_pedido") == 4, "sem limite, o agente chamou a mesma ferramenta 4 vezes")
mostrar(controle.limite_ferramenta)
Armazem.reset()
with falhas.injetar("resultado_ambiguo", alvo="buscar_pedido"):
    correcao = atender(s.correcao, s.caso, roteiro=s.roteiro(s.correcao))
resumo(correcao)
verificar(contar_chamadas(correcao.trajetoria, "buscar_pedido") <= 3 and correcao.status == "escalado", "com limite: no máximo 3 chamadas e escalou")

chamadas de buscar_pedido (sem limite): 4


agente,ferramenta,argumentos,efeito
resolucao,buscar_pedido,{'pedido_id': '4821'},✅ com efeito
resolucao,buscar_pedido,{'pedido_id': '4821'},✅ com efeito
resolucao,buscar_pedido,{'pedido_id': '4821'},✅ com efeito
resolucao,buscar_pedido,{'pedido_id': '4821'},⛔ sem efeito
resolucao,escalar_para_humano,"{'motivo': 'resultado inconsistente', 'resumo': '4821'}",✅ com efeito


## N3.5 — O processo caiu com um reembolso pendente

Caso `C09`. O reembolso de R$ 120 aguarda aprovação. O processo cai. A **fila de aprovação** está no armazém e sempre sobrevive; a **conversa** só sobrevive se a sessão estiver num banco.

In [10]:
s = preparar_secao("N3.5")
a = atender(s.falha, s.caso, roteiro=s.roteiro(s.falha))
verificar(a.status == "aguardando_aprovacao", "pendência criada")
reiniciar_runner(a)
print("estado da conversa depois da queda (sessão em memória):", estado_persistido(a))
try:
    continuar(a, "E aí, aprovaram?")
except RuntimeError as exc:
    print("continuar a conversa falhou:", exc)
verificar(estado_persistido(a) is None, "sem persistência, a conversa se perdeu")

estado da conversa depois da queda (sessão em memória): None
continuar a conversa falhou: runner reiniciado e sem persistência: a conversa em memória foi perdida


In [11]:
Armazem.reset()
b = atender(s.correcao, s.caso, roteiro=s.roteiro(s.correcao))
reiniciar_runner(b)
persistido = estado_persistido(b)
print("estado da conversa depois da queda (SQLite):", {k: persistido[k] for k in ("cliente_id", "pedido_id") if k in persistido})
verificar(persistido is not None, "com persistência, a sessão sobreviveu à queda")
fechado = aprovar(b)
verificar(total("9001") == 120.0 and fechado.status == "concluido", "o supervisor aprova depois da queda: exatamente um reembolso de R$ 120")

estado da conversa depois da queda (SQLite): {'cliente_id': '109', 'pedido_id': '9001'}


## N3.6 — Reembolso em dobro

Caso `C03`. O reembolso é feito, mas a **confirmação se perde** (timeout depois do efeito). O agente vê um erro e chama de novo. Frameworks e filas garantem *pelo menos uma vez*; *exatamente uma vez* é responsabilidade do sistema: **idempotência**.

In [12]:
s = preparar_secao("N3.6")
with falhas.injetar("timeout_apos_efeito", alvo="registrar_reembolso"):
    falha = atender(s.falha, s.caso, roteiro=s.roteiro(s.falha))
resumo(falha)
print("total reembolsado no pedido 6002 (cobrança de R$ 42,90):", total("6002"))
verificar(abs(total("6002") - 85.8) < 0.01, "sem idempotência: o cliente foi reembolsado duas vezes")

agente,ferramenta,argumentos,efeito
resolucao,registrar_reembolso,"{'pedido_id': '6002', 'valor': 42.9, 'motivo': 'cobranca'}",⛔ sem efeito
resolucao,registrar_reembolso,"{'pedido_id': '6002', 'valor': 42.9, 'motivo': 'cobranca'}",✅ com efeito


total reembolsado no pedido 6002 (cobrança de R$ 42,90): 85.8


In [13]:
mostrar(ferramentas.efetivar_reembolso)
Armazem.reset()
with falhas.injetar("timeout_apos_efeito", alvo="registrar_reembolso"):
    correcao = atender(s.correcao, s.caso, roteiro=s.roteiro(s.correcao))
resumo(correcao)
verificar(abs(total("6002") - 42.9) < 0.01, "com idempotência: a segunda chamada devolveu o resultado da primeira, um único reembolso")

agente,ferramenta,argumentos,efeito
resolucao,registrar_reembolso,"{'pedido_id': '6002', 'valor': 42.9, 'motivo': 'cobranca'}",⛔ sem efeito
resolucao,registrar_reembolso,"{'pedido_id': '6002', 'valor': 42.9, 'motivo': 'cobranca'}",✅ com efeito


## N3.7 — Recapitulação

| Falha injetada | Reação do sistema | Onde se vê |
|---|---|---|
| `buscar_pedido` fora do ar | erro vira resultado; escala | span da ferramenta com erro |
| 429 do modelo (2×) | retry com espera | tentativas por chamada |
| modelo primário fora | fallback para o secundário | `modelos_usados` |
| ferramenta devolve sempre "incompleto" | limite de 3 chamadas; escala | contagem na trajetória |
| queda do processo | fila de aprovação e sessão sobrevivem | `estado_persistido` |
| confirmação perdida após o efeito | idempotência: um único reembolso | total no armazém |

Não implementado aqui: métricas agregadas do Phoenix (taxa de erro de ferramenta, de escalonamento, latência).

**Ponte para o NB04:** o sistema reage bem *a estes casos*. Como garantir que continua reagindo depois da próxima mudança?